# Lab 10 — Distributed Machine Learning with MLlib
### From the DataFrame to a model in production

With clean data (Lab 7), often arriving in real time (Lab 9), we want to **predict** something. MLlib is Spark's machine learning library. It trains models on distributed DataFrames, with the same pipeline logic as scikit-learn.

**Problem for this lab:** predict which customers will cancel (*churn*) so we can act before it happens.

**Objectives**
- Understand Transformer, Estimator and Pipeline
- Prepare features: nulls, categories and vectors
- Train and evaluate a classification model
- Choose the decision threshold according to the business
- Tune hyperparameters with cross-validation
- Save the model and use it in batch and streaming

---
## 1. Theory

### When to use MLlib

| | scikit-learn | Spark MLlib |
|-|-------------|-------------|
| Data | Fits in the memory of one machine | Distributed, no practical limit |
| Input | NumPy / Pandas | Spark DataFrame |
| Features | A matrix `X` | **A single vector-type column** (`features`) |
| Algorithms | Very broad | The main ones: linear, trees, ensembles, clustering, ALS |
| Integration | Notebook, APIs | Same pipeline as ETL and streaming |

Rule of thumb: if the training data fits on one machine, scikit-learn is usually simpler. MLlib is worth it when the volume is large or when the model has to run inside Spark pipelines.

### The three central concepts

| Concept | What it is | Method | Examples |
|----------|---------|--------|----------|
| **Transformer** | Takes a DataFrame and returns another with new columns | `.transform(df)` | `VectorAssembler`, a trained model |
| **Estimator** | Learns something from the data and produces a Transformer | `.fit(df)` | `StringIndexer`, `StandardScaler`, `LogisticRegression` |
| **Pipeline** | Sequence of stages; it is itself an Estimator | `.fit(df)` produces a `PipelineModel` | |

```
                       Pipeline.fit(train)
  ┌──────────────┬──────────────┬───────────────┬──────────────┬──────────────────┐
  │ Imputer      │ StringIndexer│ OneHotEncoder │ Assembler    │ LogisticRegression│  Estimators
  └──────┬───────┴──────┬───────┴───────┬───────┴──────┬───────┴─────────┬────────┘
         ▼              ▼               ▼              ▼                 ▼
  ┌──────────────┬──────────────┬───────────────┬──────────────┬──────────────────┐
  │ ImputerModel │ IndexerModel │ EncoderModel  │ Assembler    │ LRModel          │  PipelineModel
  └──────────────┴──────────────┴───────────────┴──────────────┴──────────────────┘
                       PipelineModel.transform(new_data) -> predictions
```

The most important point: **everything learned from the data (means, categories, scales) is fitted on the training set only**. If the mean used to fill nulls is computed with the test data, the evaluation becomes optimistic (*data leakage*). The Pipeline guarantees this automatically.

---
## 2. Setup and data

In [1]:
# !pip install -q pyspark==4.0.1 numpy

In [2]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import vector_to_array

spark = (
    SparkSession.builder
    .appName("Lab 10 - MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 18:43:01 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/11/04 18:43:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 18:43:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/04 18:43:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.0.1


We generate 200 thousand synthetic customers. The churn probability follows a known rule (more support tickets and a basic plan increase the risk; tenure, orders and a good rating decrease it). That way we can check whether the model learned the right pattern.

We also add **null values** to `last_rating` and `monthly_spend`, as happens in real data.

In [3]:
N = 200_000
REGIONS = ["North", "Northeast", "Midwest", "Southeast", "South"]

def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.rand(seed) * len(values)).cast("int") + 1)

base = (
    spark.range(N).withColumnRenamed("id", "customer_id")
    .withColumn("tenure_months",   (F.rand(1) * 71 + 1).cast("int"))
    .withColumn("plan",            pick(["Basic", "Standard", "Premium"], 2))
    .withColumn("region",          pick(REGIONS, 3))
    .withColumn("monthly_spend",   F.round(
        F.when(F.col("plan") == "Basic", 40).when(F.col("plan") == "Standard", 120).otherwise(300)
        * (0.5 + F.rand(4)), 2))
    .withColumn("n_orders_90d",    (F.rand(5) * 15).cast("int"))
    .withColumn("support_tickets", (F.pow(F.rand(6), 2) * 9).cast("int"))
    .withColumn("has_discount",    F.rand(7) < 0.3)
    .withColumn("last_rating",     (F.rand(8) * 5 + 1).cast("int"))
)

logit = (
    0.8
    - 0.045 * F.col("tenure_months")
    + 0.45  * F.col("support_tickets")
    - 0.10  * F.col("n_orders_90d")
    - 0.45  * (F.col("last_rating") - 3)
    + F.when(F.col("plan") == "Basic", 0.5).when(F.col("plan") == "Premium", -0.6).otherwise(0.0)
    - 0.5   * F.col("has_discount").cast("int")
    + 0.6   * F.randn(9)
)

customers = (
    base
    .withColumn("churn", (F.rand(10) < 1 / (1 + F.exp(-logit))).cast("int"))
    .withColumn("last_rating",   F.when(F.rand(11) < 0.06, None).otherwise(F.col("last_rating")))
    .withColumn("monthly_spend", F.when(F.rand(12) < 0.03, None).otherwise(F.col("monthly_spend")))
)

customers.write.mode("overwrite").parquet("/tmp/lab10/customers")
customers = spark.read.parquet("/tmp/lab10/customers")
customers.show(5)

+-----------+-------------+--------+---------+-------------+------------+---------------+------------+-----------+-----+
|customer_id|tenure_months|    plan|   region|monthly_spend|n_orders_90d|support_tickets|has_discount|last_rating|churn|
+-----------+-------------+--------+---------+-------------+------------+---------------+------------+-----------+-----+
|      33333|            2|   Basic|Northeast|        39.35|          10|              0|        true|          2|    1|
|      33334|            7| Premium|  Midwest|       370.26|           6|              5|        true|          5|    1|
|      33335|           55|Standard|Southeast|         90.4|           0|              3|       false|          4|    0|
|      33336|           39| Premium|Southeast|       279.38|           3|              8|       false|          1|    1|
|      33337|           24| Premium|  Midwest|       375.69|          14|              0|        true|          1|    0|
+-----------+-------------+-----

### 2.1 Quick exploration

In [4]:
customers.select(F.round(F.avg("churn") * 100, 1).alias("churn_pct")).show()

customers.groupBy("plan").agg(
    F.count("*").alias("customers"),
    F.round(F.avg("churn") * 100, 1).alias("churn_pct")
).orderBy("churn_pct").show()

customers.groupBy("support_tickets").agg(
    F.round(F.avg("churn") * 100, 1).alias("churn_pct")
).orderBy("support_tickets").show()

print("Nulls:")
customers.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                  for c in ["last_rating", "monthly_spend"]]).show()

+---------+
|churn_pct|
+---------+
|     40.8|
+---------+

+--------+---------+---------+
|    plan|customers|churn_pct|
+--------+---------+---------+
| Premium|    66712|     32.4|
|Standard|    66773|     41.2|
|   Basic|    66515|     48.9|
+--------+---------+---------+

+---------------+---------+
|support_tickets|churn_pct|
+---------------+---------+
|              0|     22.2|
|              1|     28.3|
|              2|     35.7|
|              3|     43.5|
|              4|     51.6|
|              5|     59.4|
|              6|     67.3|
|              7|     74.1|
|              8|     80.0|
+---------------+---------+

Nulls:
+-----------+-------------+
|last_rating|monthly_spend|
+-----------+-------------+
|      11966|         5906|
+-----------+-------------+



---
## 3. Train and test

The split comes **before** any feature preparation. `randomSplit` with a `seed` makes the split reproducible.

In [5]:
train, test = customers.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"Train: {train.count():,} | Test: {test.count():,}")

Train: 160,091 | Test: 39,909


---
## 4. Feature preparation

MLlib algorithms expect **a single numeric vector column**. Each type of column needs its own treatment:

| Column | Problem | Stage |
|--------|----------|---------|
| `last_rating`, `monthly_spend` | Nulls | `Imputer` (median) |
| `plan`, `region` | Text | `StringIndexer` (text -> index) + `OneHotEncoder` (index -> binary vector) |
| All | Several columns | `VectorAssembler` (combines into a vector) |
| Numeric | Different scales | `StandardScaler` (important for linear models) |

Let's see the effect of each stage separately on a few rows.

In [6]:
sample = train.limit(5)

imputer = Imputer(strategy="median",
                  inputCols=["last_rating", "monthly_spend"],
                  outputCols=["last_rating_imp", "monthly_spend_imp"])
step1 = imputer.fit(train).transform(sample)
step1.select("last_rating", "last_rating_imp", "monthly_spend", "monthly_spend_imp").show()

+-----------+---------------+-------------+-----------------+
|last_rating|last_rating_imp|monthly_spend|monthly_spend_imp|
+-----------+---------------+-------------+-----------------+
|          2|              2|        39.35|            39.35|
|          5|              5|       370.26|           370.26|
|          1|              1|       279.38|           279.38|
|          1|              1|       375.69|           375.69|
|          3|              3|       109.48|           109.48|
+-----------+---------------+-------------+-----------------+



In [7]:
indexer = StringIndexer(inputCols=["plan", "region"], outputCols=["plan_idx", "region_idx"],
                        handleInvalid="keep")          # new category in the future -> extra index
encoder = OneHotEncoder(inputCols=["plan_idx", "region_idx"], outputCols=["plan_vec", "region_vec"])

idx_model = indexer.fit(train)
print("Category order (from most to least frequent):", idx_model.labelsArray)

step2 = encoder.fit(idx_model.transform(train)).transform(idx_model.transform(sample))
step2.select("plan", "plan_idx", "plan_vec", "region", "region_vec").show(truncate=False)

Category order (from most to least frequent): [['Premium', 'Standard', 'Basic'], ['South', 'Southeast', 'Midwest', 'Northeast', 'North']]
+--------+--------+-------------+---------+-------------+
|plan    |plan_idx|plan_vec     |region   |region_vec   |
+--------+--------+-------------+---------+-------------+
|Basic   |2.0     |(3,[2],[1.0])|Northeast|(5,[3],[1.0])|
|Premium |0.0     |(3,[0],[1.0])|Midwest  |(5,[2],[1.0])|
|Premium |0.0     |(3,[0],[1.0])|Southeast|(5,[1],[1.0])|
|Premium |0.0     |(3,[0],[1.0])|Midwest  |(5,[2],[1.0])|
|Standard|1.0     |(3,[1],[1.0])|South    |(5,[0],[1.0])|
+--------+--------+-------------+---------+-------------+



`(3,[1],[1.0])` is a **sparse vector**: size 3, value 1.0 at position 1, zeros elsewhere.

Why size 3 if there are 3 plans? With `handleInvalid="keep"`, `StringIndexer` reserves an extra index for unknown categories (4 in total). `OneHotEncoder` drops the last position (`dropLast=True`), which is precisely the "unknown" one. A new category becomes an all-zero vector.

In [8]:
NUMERIC = ["tenure_months", "monthly_spend_imp", "n_orders_90d", "support_tickets", "last_rating_imp"]

assembler = VectorAssembler(inputCols=NUMERIC + ["has_discount", "plan_vec", "region_vec"],
                            outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False)

prep = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler]).fit(train)
prep.transform(sample).select("customer_id", "features").show(3, truncate=False)

+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|customer_id|features                                                                                                                                                                     |
+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|33333      |(14,[0,1,2,4,5,8,12],[0.09761743173080811,0.32815511501943667,2.3129241591698517,1.4570098078641665,2.1822279142728385,2.1221270315598786,2.502375151166683])                |
|33334      |(14,[0,1,2,3,4,5,6,11],[0.3416610110578284,3.0877436565971186,1.387754495501911,1.908562301064785,3.642524519660416,2.1822279142728385,2.1203092930769682,2.498127448835056])|
|33336      |(14,[0,1,2,3,4,6,10],[1.9035399187507582,2.3298

---
## 5. First model: logistic regression

The full pipeline is the preparation plus the model. A single `fit` fits all the stages, in order, using only the training set.

In [9]:
lr = LogisticRegression(featuresCol="features", labelCol="churn", maxIter=50)
lr_pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler, lr])

t0 = time.perf_counter()
lr_model = lr_pipeline.fit(train)
print(f"Training: {time.perf_counter() - t0:.1f} s")

pred = lr_model.transform(test)
pred.select("customer_id", "churn", "probability", "prediction").show(5, truncate=False)

Training: 2.3 s
+-----------+-----+----------------------------------------+----------+
|customer_id|churn|probability                             |prediction|
+-----------+-----+----------------------------------------+----------+
|33335      |0    |[0.6711942882716843,0.32880571172831574]|0.0       |
|33339      |0    |[0.11116990372080929,0.8888300962791907]|1.0       |
|33341      |1    |[0.15168798240185186,0.8483120175981481]|1.0       |
|33346      |1    |[0.33894464016172376,0.6610553598382762]|1.0       |
|33352      |0    |[0.7561792884851826,0.24382071151481743]|0.0       |
+-----------+-----+----------------------------------------+----------+
only showing top 5 rows


Three new columns:
- `rawPrediction`: the model's raw score.
- `probability`: vector `[P(class 0), P(class 1)]`.
- `prediction`: the final class, using the default threshold of 0.5.

### 5.1 Evaluation

| Metric | Question | When it matters |
|---------|----------|----------------|
| **AUC-ROC** | Does the model rank customers well by risk? | Comparing models, regardless of threshold |
| **AUC-PR** | Same, focused on the positive class | Imbalanced classes |
| **Precision** | Of those I flagged as churn, how many were? | Cost of contacting a customer unnecessarily |
| **Recall** | Of those who churned, how many did I find? | Cost of losing a customer |
| **Accuracy** | How many did I get right overall? | Misleading with imbalanced classes |

In [10]:
def evaluate(predictions, name="model"):
    auc_roc = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC").evaluate(predictions)
    auc_pr  = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderPR").evaluate(predictions)
    mc = lambda m, **kw: MulticlassClassificationEvaluator(labelCol="churn", metricName=m, **kw).evaluate(predictions)
    print(f"{name:<22} AUC-ROC={auc_roc:.3f}  AUC-PR={auc_pr:.3f}  "
          f"accuracy={mc('accuracy'):.3f}  precision={mc('precisionByLabel', metricLabel=1.0):.3f}  "
          f"recall={mc('recallByLabel', metricLabel=1.0):.3f}")

evaluate(pred, "Logistic regression")

print("\nConfusion matrix:")
pred.groupBy("churn").pivot("prediction", [0.0, 1.0]).count().orderBy("churn").show()

Logistic regression    AUC-ROC=0.831  AUC-PR=0.778  accuracy=0.760  precision=0.731  recall=0.655

Confusion matrix:
+-----+-----+-----+
|churn|  0.0|  1.0|
+-----+-----+-----+
|    0|19641| 3932|
|    1| 5636|10700|
+-----+-----+-----+



### 5.2 What the model learned

The coefficients indicate each feature's direction: positive increases the churn risk. Since the features were standardized, the magnitudes are comparable.

`VectorAssembler` stores the name of each vector position in the column's **metadata**.

In [11]:
def feature_names(df, col="features_raw"):
    attrs = df.schema[col].metadata["ml_attr"]["attrs"]
    return [a["name"] for a in sorted((a for group in attrs.values() for a in group), key=lambda a: a["idx"])]

names = feature_names(pred)
coefs = lr_model.stages[-1].coefficients.toArray()

(spark.createDataFrame([(n, float(c)) for n, c in zip(names, coefs)], ["feature", "coef"])
 .orderBy(F.desc(F.abs("coef")))
 .show(truncate=False))

+--------------------+----------------------+
|feature             |coef                  |
+--------------------+----------------------+
|support_tickets     |1.100218100792409     |
|tenure_months       |-0.8628476537300825   |
|last_rating_imp     |-0.5680250415357407   |
|n_orders_90d        |-0.3992405398553087   |
|plan_vec_Premium    |-0.24513075997400424  |
|has_discount        |-0.2271654838977809   |
|plan_vec_Basic      |0.22714018241087144   |
|plan_vec_Standard   |0.018195429488176172  |
|region_vec_South    |0.007838364602923325  |
|monthly_spend_imp   |-0.007124135838575218 |
|region_vec_North    |-0.003968473783101264 |
|region_vec_Northeast|-0.0022065169237304835|
|region_vec_Midwest  |-0.0019280759411731824|
|region_vec_Southeast|2.3640290575588398E-4 |
+--------------------+----------------------+



The model recovered the rule used to generate the data: `support_tickets` and the basic plan increase the risk; `tenure_months`, `last_rating`, `n_orders_90d` and a discount decrease it; the region barely matters.

### 5.3 Choosing the decision threshold

A threshold of 0.5 is rarely ideal. The choice is a **business decision**: if calling a customer costs little and losing a customer costs a lot, we accept more false positives to increase recall.

In [12]:
scored = pred.select("churn", vector_to_array("probability")[1].alias("p_churn")).cache()

rows = []
for th in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    r = scored.agg(
        F.sum(((F.col("p_churn") >= th) & (F.col("churn") == 1)).cast("int")).alias("tp"),
        F.sum(((F.col("p_churn") >= th) & (F.col("churn") == 0)).cast("int")).alias("fp"),
        F.sum(((F.col("p_churn") <  th) & (F.col("churn") == 1)).cast("int")).alias("fn"),
    ).first()
    precision = r.tp / (r.tp + r.fp) if r.tp + r.fp else 0.0
    recall    = r.tp / (r.tp + r.fn)
    rows.append((th, r.tp + r.fp, round(precision, 3), round(recall, 3)))

spark.createDataFrame(rows, ["threshold", "customers_contacted", "precision", "recall"]).show()

+---------+-------------------+---------+------+
|threshold|customers_contacted|precision|recall|
+---------+-------------------+---------+------+
|      0.2|              27307|    0.549| 0.918|
|      0.3|              22384|    0.614| 0.841|
|      0.4|              18217|    0.674| 0.752|
|      0.5|              14632|    0.731| 0.655|
|      0.6|              11374|    0.784| 0.546|
|      0.7|               8218|    0.835|  0.42|
+---------+-------------------+---------+------+



To apply another threshold on the model itself: `lr_model.stages[-1].setThreshold(0.3)`.

---
## 6. Second model: Random Forest

Tree models capture interactions and non-linearities, and they do not need scaling. We reuse the same preparation stages.

| | Logistic regression | Random Forest | Gradient-Boosted Trees |
|-|--------------------|---------------|------------------------|
| Relationships | Linear | Non-linear | Non-linear |
| Interpretation | Coefficients | Feature importance | Feature importance |
| Training | Fast | Parallel (independent trees) | Sequential (slower) |
| Overfitting risk | Low | Medium | Higher, requires tuning |

In [13]:
rf = RandomForestClassifier(featuresCol="features_raw", labelCol="churn",
                            numTrees=60, maxDepth=8, seed=42)
rf_pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, rf])

t0 = time.perf_counter()
rf_model = rf_pipeline.fit(train)
print(f"Training: {time.perf_counter() - t0:.1f} s")

rf_pred = rf_model.transform(test)
evaluate(pred,    "Logistic regression")
evaluate(rf_pred, "Random Forest")

importances = rf_model.stages[-1].featureImportances.toArray()
(spark.createDataFrame([(n, float(v)) for n, v in zip(feature_names(rf_pred), importances)],
                       ["feature", "importance"])
 .orderBy(F.desc("importance")).show(6, truncate=False))

Training: 5.1 s
Logistic regression    AUC-ROC=0.831  AUC-PR=0.778  accuracy=0.760  precision=0.731  recall=0.655
Random Forest          AUC-ROC=0.818  AUC-PR=0.760  accuracy=0.751  precision=0.733  recall=0.615
+-----------------+--------------------+
|feature          |importance          |
+-----------------+--------------------+
|support_tickets  |0.5277316853091433  |
|tenure_months    |0.2588765611018186  |
|last_rating_imp  |0.09738253187485277 |
|n_orders_90d     |0.0402261812214249  |
|plan_vec_Premium |0.028123164345379315|
|monthly_spend_imp|0.01930778631583138 |
+-----------------+--------------------+
only showing top 6 rows


Here logistic regression ties or beats the Random Forest. That is no surprise: the data was generated by a linear rule. **The more complex model is not automatically the best**; always compare against a simple baseline.

---
## 7. Exercises

In [14]:
# ── Exercise 1 ──────────────────────────────────────────────────────────────
# Create the derived feature spend_per_order = monthly_spend_imp / (n_orders_90d + 1)
# and add it to the logistic regression pipeline. Does the AUC improve? Why?
# Hint: use SQLTransformer to create the column INSIDE the pipeline:
#   from pyspark.ml.feature import SQLTransformer
#   SQLTransformer(statement="SELECT *, ... AS spend_per_order FROM __THIS__")

# Your solution:
from pyspark.ml.feature import SQLTransformer

ratio = SQLTransformer(statement=
    "SELECT *, monthly_spend_imp / (n_orders_90d + 1) AS spend_per_order FROM __THIS__")
assembler_v2 = VectorAssembler(inputCols=NUMERIC + ["spend_per_order", "has_discount", "plan_vec", "region_vec"],
                               outputCol="features_raw")

lr_v2 = Pipeline(stages=[imputer, ratio, indexer, encoder, assembler_v2, scaler, lr]).fit(train)
evaluate(pred, "LR original")
evaluate(lr_v2.transform(test), "LR + spend_per_order")
# It does not improve: churn was generated with no relation to this ratio.
# Feature engineering only helps when it captures a real pattern in the problem.

LR original            AUC-ROC=0.831  AUC-PR=0.777  accuracy=0.760  precision=0.731  recall=0.655
LR + spend_per_order   AUC-ROC=0.831  AUC-PR=0.777  accuracy=0.760  precision=0.731  recall=0.655


In [15]:
# ── Exercise 2 ──────────────────────────────────────────────────────────────
# Train a GBTClassifier (maxIter=30, maxDepth=4) with the same preparation as the Random Forest.
# Compare AUC-ROC and training time with the other two models.

# Your solution:
gbt = GBTClassifier(featuresCol="features_raw", labelCol="churn", maxIter=30, maxDepth=4, seed=42)

t0 = time.perf_counter()
gbt_model = Pipeline(stages=[imputer, indexer, encoder, assembler, gbt]).fit(train)
print(f"GBT training: {time.perf_counter() - t0:.1f} s")

evaluate(pred,                        "Logistic regression")
evaluate(rf_pred,                     "Random Forest")
evaluate(gbt_model.transform(test),   "GBT")

GBT training: 6.4 s
Logistic regression    AUC-ROC=0.831  AUC-PR=0.777  accuracy=0.760  precision=0.731  recall=0.655
Random Forest          AUC-ROC=0.818  AUC-PR=0.760  accuracy=0.751  precision=0.733  recall=0.615
GBT                    AUC-ROC=0.825  AUC-PR=0.768  accuracy=0.755  precision=0.727  recall=0.644


In [16]:
# ── Exercise 3 ──────────────────────────────────────────────────────────────
# The retention team wants to find at least 80% of the customers who will churn.
# Find the HIGHEST threshold that guarantees recall >= 0.80 on the test set and report the precision
# and how many customers would be contacted.

# Your solution:
total_churn = scored.filter("churn = 1").count()

best_th = None
for th in [x / 100 for x in range(5, 96)]:
    tp = scored.filter((F.col("p_churn") >= th) & (F.col("churn") == 1)).count()
    if tp / total_churn >= 0.80:
        best_th = th
    else:
        break

flagged = scored.filter(F.col("p_churn") >= best_th)
tp = flagged.filter("churn = 1").count()
print(f"Threshold = {best_th:.2f} | recall = {tp / total_churn:.3f} | "
      f"precision = {tp / flagged.count():.3f} | customers contacted = {flagged.count():,}")

Threshold = 0.34 | recall = 0.807 | precision = 0.638 | customers contacted = 20,659


In [17]:
# ── Exercise 4 ──────────────────────────────────────────────────────────────
# Regression: predict monthly_spend from the other columns (without using churn).
# Use LinearRegression, drop rows with null monthly_spend and evaluate with
# RegressionEvaluator (RMSE and R2). Which feature explains most of the spending?

# Your solution:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

train_r = train.filter(F.col("monthly_spend").isNotNull())
test_r  = test.filter(F.col("monthly_spend").isNotNull())

imp_r = Imputer(strategy="median", inputCols=["last_rating"], outputCols=["last_rating_imp"])
asm_r = VectorAssembler(
    inputCols=["tenure_months", "n_orders_90d", "support_tickets", "last_rating_imp",
               "has_discount", "plan_vec", "region_vec"],
    outputCol="features_raw")
reg = LinearRegression(featuresCol="features_raw", labelCol="monthly_spend")

reg_model = Pipeline(stages=[imp_r, indexer, encoder, asm_r, reg]).fit(train_r)
reg_pred = reg_model.transform(test_r)

for m in ["rmse", "r2"]:
    print(m.upper(), "=", round(RegressionEvaluator(labelCol="monthly_spend", metricName=m).evaluate(reg_pred), 3))

(spark.createDataFrame(list(zip(feature_names(reg_pred), map(float, reg_model.stages[-1].coefficients))),
                       ["feature", "coef"])
 .orderBy(F.desc(F.abs("coef"))).show(4, truncate=False))
# The plan dominates: spending was generated as (plan value) x (random factor).

RMSE = 54.176
R2 = 0.801
+-----------------+-------------------+
|feature          |coef               |
+-----------------+-------------------+
|plan_vec_Premium |146.44141071858584 |
|plan_vec_Basic   |-113.38982521988217|
|plan_vec_Standard|-33.391698473397604|
|has_discount     |0.4607809535507308 |
+-----------------+-------------------+
only showing top 4 rows


In [18]:
spark.stop()